# Heartlink – Giả lập tải & đo metrics trên Kaggle CPU (Pha 1 baseline)

Notebook này dựng **đúng stack của repo** (Spring Boot + PostgreSQL), nạp dữ liệu giả lập, rồi chạy 7 kịch bản đo (6 mối quan tâm riêng + chat theo match, tính năng mới của repo `HEART_LINK`) của bài toán ghép đôi. Kết quả (CSV + biểu đồ + `env.json`) nằm trong `/kaggle/working/results/` để **Pha 2 chạy lại đúng notebook này** và so sánh trước/sau.

| Kịch bản | Mối quan tâm | Metric chính |
|---|---|---|
| S1 | Chi phí gợi ý tăng theo lịch sử swipe | latency GET /recommendations theo kích thước lịch sử, thời gian thực thi SQL, số block đọc |
| S2 | Tải ghi swipe | RPS, p50/p95/p99, % lỗi, CPU, theo mức đồng thời |
| S3 | Đúng đắn của match (2 lần ghi độc lập) | số vi phạm bất biến (kỳ vọng = 0), deadlock |
| S4 | Hồ sơ nổi tiếng (hot profile) | latency hot vs cold, số tiến trình chờ lock |
| S5 | Độ trễ thông báo (SSE) | delay gửi→nhận, tỉ lệ mất sự kiện, RAM/thread theo số kết nối |
| S6 | Phiên người dùng hỗn hợp | RPS và p95 tổng thể (baseline để so Pha 2) |
| S7 | Chat theo match (gửi tin khóa dòng `matches`, đọc lịch sử, polling) | RPS/p95 gửi tin rải đều vs dồn vào 1 match, đọc lịch sử theo kích thước, tải polling, đúng đắn khi unmatch đồng thời |

**Trước khi chạy:** bật *Internet* trong Settings của notebook (cần để apt/Maven/git). Chạy tuần tự từ trên xuống. Đặt `QUICK = True` ở ô cấu hình để chạy thử nhanh vài phút.

In [ ]:
import os, sys, re, json, time, math, random, shutil, subprocess, threading, asyncio, platform, resource
from pathlib import Path

QUICK = bool(int(os.environ.get("HL_QUICK", "0")))   # True = chạy thử nhanh, KHÔNG dùng số liệu này để báo cáo

ON_KAGGLE = Path("/kaggle/working").exists()
WORK    = Path("/kaggle/working") if ON_KAGGLE else Path("/tmp/hl_work")
RESULTS = WORK / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
SRC     = Path("/tmp/HEART_LINK")          # repo + build để ngoài /kaggle/working cho nhẹ output
PGDATA  = Path("/tmp/hl_pgdata")

CFG = dict(
    REPO="https://github.com/viet2005-68/HEART_LINK.git", BRANCH="main",
    REF=None,                                                   # ghim commit/tag để đo lại đúng bản cần (None = đầu nhánh BRANCH)
    PG_PORT=5433, APP_PORT=8080,
    JVM_OPTS="-Xms1g -Xmx2g",                                   # CỐ ĐỊNH cho cả Pha 1 và Pha 2
    PG_OPTS="-c shared_buffers=512MB -c max_connections=200",   # CỐ ĐỊNH
    N_USERS=50_000,
    DURATION=30, WARMUP=5,                                      # giây / mỗi mức tải (warm-up bị loại khỏi số liệu)
    S1_LEVELS=[0, 100, 1000, 5000, 10000, 20000], S1_PROBES=16, S1_CONC=8,
    S2_CONC=[1, 4, 16, 32, 64],
    S3_PAIRS=1000, S3_DUP=300, S3_STAR=150, S3_REPEATS=3,
    S4_CONC=32, S4_ZIPF=1.2, S4_HOT_K=10,
    S5_CONNS=[100, 1000, 3000], S5_PAIRS=100, S5_RATE=20,
    S6_CONC=[8, 32, 64],
    S7_MATCHES=1000, S7_CONC=32, S7_HIST=[0, 1000, 10000], S7_POLLERS=[200, 1000], S7_POLL_S=2.0, S7_CANCEL=100,
    SEED=42,
)
if QUICK:
    CFG.update(N_USERS=3000, DURATION=4, WARMUP=1, S1_LEVELS=[0, 100, 500], S1_PROBES=4, S1_CONC=4,
               S2_CONC=[1, 8], S3_PAIRS=60, S3_DUP=20, S3_STAR=30, S3_REPEATS=1,
               S4_CONC=8, S5_CONNS=[20, 100], S5_PAIRS=10, S5_RATE=10, S6_CONC=[4],
               S7_MATCHES=60, S7_CONC=8, S7_HIST=[0, 500], S7_POLLERS=[20, 60], S7_POLL_S=1.0, S7_CANCEL=20)

JWT_SECRET = "heartlink-bench-secret-0123456789abcdef"   # 39 byte -> HS256
print("QUICK =", QUICK, "| WORK =", WORK)
print(json.dumps({k: v for k, v in CFG.items() if k not in ("REPO",)}, indent=1)[:900])

## 1. Cài môi trường (JDK 21, PostgreSQL, thư viện Python)
Kaggle notebook thường **không có Docker daemon**, nên ta chạy app và PostgreSQL trực tiếp trong container notebook (cùng phiên bản với repo: Java 21, PostgreSQL 16). Dockerfile/compose vẫn nằm trong repo để đáp ứng đề bài; chỉ phần *đo* chạy ngoài Docker. Nên ghi rõ điều này trong báo cáo.

In [ ]:
def sh(cmd, check=True, timeout=None):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    if check and r.returncode != 0:
        raise RuntimeError(f"$ {cmd}\n--- stdout ---\n{r.stdout[-3000:]}\n--- stderr ---\n{r.stderr[-3000:]}")
    return r.stdout.strip()

PIP = "pip install -q aiohttp psycopg2-binary pyjwt psutil pandas matplotlib numpy"
if subprocess.run(PIP, shell=True, capture_output=True).returncode != 0:      # môi trường PEP 668
    sh(PIP + " --break-system-packages", timeout=900)

# ---- Java 21 ----
def java_major():
    # cần JDK (có javac) chứ không chỉ JRE, vì Maven phải biên dịch
    r = subprocess.run("javac -version", shell=True, text=True, capture_output=True)
    m = re.search(r'javac (\d+)', r.stderr + r.stdout)
    return int(m.group(1)) if m else 0

def use_jdk(home):
    os.environ["JAVA_HOME"] = str(home)
    os.environ["PATH"] = f"{home}/bin:" + os.environ["PATH"]

if java_major() < 21:
    sh("apt-get update -qq", check=False)
    sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jdk-headless", check=False, timeout=900)
    for cand in sorted(Path("/usr/lib/jvm").glob("*21*")):
        if (cand / "bin/java").exists(): use_jdk(cand); break
if java_major() < 21:     # dự phòng: tải Temurin 21
    sh("mkdir -p /opt/jdk21 && curl -fsSL 'https://api.adoptium.net/v3/binary/latest/21/ga/linux/x64/jdk/hotspot/normal/eclipse' | tar -xz -C /opt/jdk21 --strip-components=1", timeout=900)
    use_jdk("/opt/jdk21")
assert java_major() >= 21, "Không cài được JDK 21"
print("Java:", sh("java -version 2>&1 | head -1"), "|", sh("javac -version 2>&1"))

# ---- PostgreSQL ----
def pg_bin():
    c = sorted(Path("/usr/lib/postgresql").glob("*/bin/initdb"), key=lambda p: int(p.parent.parent.name))
    return c[-1].parent if c else None

if pg_bin() is None or int(pg_bin().parent.name) < 16:
    sh("apt-get update -qq", check=False)
    sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql-16 postgresql-contrib", check=False, timeout=900)
if pg_bin() is None or int(pg_bin().parent.name) < 16:      # thêm kho PGDG nếu bản Ubuntu chưa có PG16
    sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq curl ca-certificates gnupg", check=False)
    sh("install -d /usr/share/postgresql-common/pgdg && curl -fsSL -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc", check=False)
    sh('. /etc/os-release && echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt ${VERSION_CODENAME}-pgdg main" > /etc/apt/sources.list.d/pgdg.list', check=False)
    sh("apt-get update -qq", check=False)
    sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql-16 postgresql-contrib", check=False, timeout=900)
if pg_bin() is None:
    sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql postgresql-contrib", timeout=900)
PGBIN = pg_bin()
print("PostgreSQL bin:", PGBIN)

## 2. Build ứng dụng, khởi động PostgreSQL + app

In [ ]:
# ---- clone + build ----
if not (SRC / ".git").exists():
    if CFG["REF"]:      # clone đầy đủ để checkout được commit cũ (vd. "b7fd0cb" = baseline Pha 1)
        sh(f"git clone {CFG['REPO']} {SRC} && git -C {SRC} checkout --quiet {CFG['REF']}", timeout=600)
    else:
        sh(f"git clone --depth 1 --branch {CFG['BRANCH']} {CFG['REPO']} {SRC}", timeout=600)
APP_DIR = SRC / "heartlink"
COMMIT = sh(f"cd {SRC} && git rev-parse --short HEAD")
print("commit:", COMMIT)
sh(f"cd {APP_DIR} && chmod +x mvnw && ./mvnw -q -DskipTests package", timeout=2400)
JAR = next((APP_DIR / "target").glob("heartlink-*.jar"))
print("jar:", JAR.name)

# ---- nâng giới hạn file descriptor (cần cho S5: nhiều kết nối SSE) ----
try:
    soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (hard, hard))
except Exception as e:
    print("ulimit:", e)
print("RLIMIT_NOFILE =", resource.getrlimit(resource.RLIMIT_NOFILE))

# ---- dọn tiến trình của lần chạy trước (nếu chạy lại ô này) ----
import psutil
if "APP" in globals() and APP.poll() is None:
    APP.terminate()
    try: APP.wait(timeout=20)
    except Exception: APP.kill()
if (PGDATA / "postmaster.pid").exists():
    sh(f"su postgres -c '{PGBIN}/pg_ctl -D {PGDATA} -m immediate stop'", check=False)
try:
    for cn in psutil.net_connections(kind="inet"):
        if cn.status == "LISTEN" and cn.laddr.port in (CFG["APP_PORT"], CFG["PG_PORT"]) and cn.pid:
            psutil.Process(cn.pid).kill()
    time.sleep(2)
except Exception as e:
    print("Không quét được cổng:", e)

# ---- PostgreSQL riêng (cổng 5433, không đụng cluster mặc định) ----
sh("id postgres >/dev/null 2>&1 || useradd -m postgres", check=False)
sh(f"rm -rf {PGDATA} && mkdir -p {PGDATA} && chown postgres {PGDATA}")
sh(f"su postgres -c '{PGBIN}/initdb -D {PGDATA} -A trust -E UTF8 --no-locale'")
sh(f"su postgres -c \"{PGBIN}/pg_ctl -D {PGDATA} -o '-p {CFG['PG_PORT']} -c listen_addresses=127.0.0.1 {CFG['PG_OPTS']}' -l /tmp/hl_pg.log -w start\"", timeout=120)
sh(f"su postgres -c '{PGBIN}/createdb -h 127.0.0.1 -p {CFG['PG_PORT']} heartlink'")
PG_PID = int(Path(PGDATA / "postmaster.pid").read_text().splitlines()[0])
print("postgres pid:", PG_PID)

# ---- khởi động app ----
BASE = f"http://127.0.0.1:{CFG['APP_PORT']}"
env = dict(os.environ,
           SPRING_DATASOURCE_URL=f"jdbc:postgresql://127.0.0.1:{CFG['PG_PORT']}/heartlink",
           SPRING_DATASOURCE_USERNAME="postgres", SPRING_DATASOURCE_PASSWORD="123456",
           JWT_SECRET=JWT_SECRET, SERVER_PORT=str(CFG["APP_PORT"]))
APP_LOG = open(WORK / "app.log", "w")
APP = subprocess.Popen(["java", *CFG["JVM_OPTS"].split(), "-jar", str(JAR)], env=env, stdout=APP_LOG, stderr=subprocess.STDOUT)

import urllib.request, psycopg2
def app_log_tail(): return (WORK / "app.log").read_text(errors="replace")[-3500:]
NEED = ("users", "swipes", "matches", "notifications", "messages")
t0 = time.time()
while True:
    if APP.poll() is not None:
        raise RuntimeError("App thoát sớm (có thể cổng 8080 đang bị chiếm hoặc không kết nối được DB):\n" + app_log_tail())
    if time.time() - t0 > 240:
        raise RuntimeError("App không sẵn sàng sau 240s:\n" + app_log_tail())
    try:
        if urllib.request.urlopen(BASE + "/v3/api-docs", timeout=2).status == 200:
            conn = psycopg2.connect(f"host=127.0.0.1 port={CFG['PG_PORT']} dbname=heartlink user=postgres password=123456")
            conn.autocommit = True; cur = conn.cursor()
            cur.execute("select " + ",".join(f"to_regclass('public.{t}') is not null" for t in NEED)); ok = all(cur.fetchone()); conn.close()
            if ok: break
    except Exception:
        pass
    time.sleep(2)
print(f"App sẵn sàng sau {time.time()-t0:.0f}s (pid {APP.pid})")

## 3. Thư viện đo: sinh tải, thống kê, giám sát tài nguyên
- **Sinh tải closed-loop** (aiohttp): N "người dùng ảo" gửi request liên tục trong `DURATION` giây, bỏ `WARMUP` giây đầu.
- **Token**: tự ký JWT HS256 bằng đúng `JWT_SECRET` của app → bỏ qua BCrypt khi đo (BCrypt là CPU-bound, sẽ làm nhiễu số liệu của các endpoint khác). Đăng nhập thật được kiểm tra riêng ở bước smoke test.
- **Giám sát** mỗi giây: CPU (số core đang dùng) của Java, PostgreSQL và chính tiến trình sinh tải; RSS; số tiến trình PostgreSQL đang **chờ lock**.
- Lưu ý: server, PostgreSQL và bộ sinh tải **dùng chung CPU** của notebook. Cột `client_cores` cho biết bộ sinh tải tốn bao nhiêu core; nếu nó chiếm phần lớn CPU thì số liệu bị chặn bởi client.

In [ ]:
import aiohttp, jwt, psutil, psycopg2
import numpy as np, pandas as pd

def dsn(): return f"host=127.0.0.1 port={CFG['PG_PORT']} dbname=heartlink user=postgres password=123456"

def db(sql, params=None, fetch=False):
    conn = psycopg2.connect(dsn())
    try:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall() if fetch else None
    finally:
        conn.close()

_tok = {}
def token(uid):
    t = _tok.get(uid)
    if t is None:
        now = int(time.time())
        t = _tok[uid] = jwt.encode({"sub": str(uid), "iat": now, "exp": now + 6 * 3600}, JWT_SECRET, algorithm="HS256")
    return t

def arun(coro):
    # chạy coroutine trong thread riêng -> dùng được cả trong Jupyter (đã có event loop) lẫn script
    box = {}
    def target():
        try: box["v"] = asyncio.run(coro)
        except BaseException as e: box["e"] = e
    th = threading.Thread(target=target); th.start(); th.join()
    if "e" in box: raise box["e"]
    return box["v"]

async def call(sess, rec, name, method, path, uid, body=None, ok=(200,), tag=""):
    t0 = time.perf_counter()
    st, data = -1, b""
    try:
        async with sess.request(method, BASE + path, json=body,
                                headers={"Authorization": "Bearer " + token(uid)}) as r:
            data = await r.read(); st = r.status
    except Exception:
        pass
    t1 = time.perf_counter()
    rec.append((t0, name, tag, (t1 - t0) * 1000.0, st, st in ok))
    return st, data

COLS = ["t0", "name", "tag", "lat_ms", "status", "ok"]

async def closed_loop(n_vu, duration, warmup, worker):
    rec = []
    t_start = time.perf_counter(); stop = t_start + warmup + duration
    conn = aiohttp.TCPConnector(limit=0)
    async with aiohttp.ClientSession(connector=conn, timeout=aiohttp.ClientTimeout(total=60)) as sess:
        async def vu(i):
            rng = random.Random(CFG["SEED"] * 1000 + i + int(time.time()) % 997)
            while time.perf_counter() < stop:
                await worker(sess, rec, rng, i)
        await asyncio.gather(*[vu(i) for i in range(n_vu)])
    df = pd.DataFrame(rec, columns=COLS)
    df["t"] = df.t0 - t_start
    return df[df.t >= warmup].copy()

LOCK_SQL = '''select count(*) filter (where wait_event_type='Lock'), count(*) filter (where state='active')
              from pg_stat_activity where datname='heartlink' and pid<>pg_backend_pid()'''

class Monitor(threading.Thread):
    def __init__(self, interval=1.0):
        super().__init__(daemon=True)
        self.app, self.pg, self.me = psutil.Process(APP.pid), psutil.Process(PG_PID), psutil.Process(os.getpid())
        self.interval, self.ev, self.samples = interval, threading.Event(), []
    @staticmethod
    def _use(procs):
        cpu = rss = 0
        for p in procs:
            try:
                t = p.cpu_times(); cpu += t.user + t.system; rss += p.memory_info().rss
            except psutil.Error: pass
        return cpu, rss
    def run(self):
        conn = psycopg2.connect(dsn()); conn.autocommit = True; cur = conn.cursor()
        prev = None
        while not self.ev.is_set():
            now = time.perf_counter()
            try: pgp = [self.pg] + self.pg.children(recursive=True)
            except psutil.Error: pgp = [self.pg]
            a, b, c = self._use([self.app]), self._use(pgp), self._use([self.me])
            try:
                cur.execute(LOCK_SQL); lw, act = cur.fetchone()
            except Exception: lw = act = 0
            if prev:
                dt = max(now - prev[0], 1e-6)
                try: thr = self.app.num_threads()
                except psutil.Error: thr = 0
                self.samples.append(dict(
                    java_cores=max(0, (a[0] - prev[1]) / dt), pg_cores=max(0, (b[0] - prev[2]) / dt),
                    client_cores=max(0, (c[0] - prev[3]) / dt),
                    java_rss_mb=a[1] / 2**20, pg_rss_mb=b[1] / 2**20,
                    lock_waiters=lw, pg_active=act, java_threads=thr))
            prev = (now, a[0], b[0], c[0])
            self.ev.wait(self.interval)
        conn.close()
    def stop(self):
        self.ev.set(); self.join(timeout=5)
    def summary(self, skip=0):
        s = pd.DataFrame(self.samples[skip:]) if len(self.samples) > skip else pd.DataFrame(self.samples)
        if s.empty: return {}
        return dict(java_cores=s.java_cores.mean(), pg_cores=s.pg_cores.mean(), client_cores=s.client_cores.mean(),
                    java_rss_mb=s.java_rss_mb.max(), pg_rss_mb=s.pg_rss_mb.max(), java_threads=s.java_threads.max(),
                    lock_waiters_mean=s.lock_waiters.mean(), lock_waiters_max=s.lock_waiters.max(),
                    pg_active_mean=s.pg_active.mean())

ROWS = []   # mỗi dòng = 1 (kịch bản, mức tải, endpoint, tag)

def pct(x, q): return float(np.percentile(x, q)) if len(x) else float("nan")

def summarize(df, scenario, level, mon, extra=None, by_tag=True):
    out = []
    groups = df.groupby(["name", "tag"]) if by_tag else df.groupby("name")
    for k, g in groups:
        name, tag = k if by_tag else (k, "all")
        row = dict(scenario=scenario, level=level, name=name, tag=tag, count=len(g),
                   rps=len(g) / CFG["DURATION"], err_pct=100.0 * (1 - g.ok.mean()),
                   mean_ms=g.lat_ms.mean(), p50_ms=pct(g.lat_ms, 50), p95_ms=pct(g.lat_ms, 95),
                   p99_ms=pct(g.lat_ms, 99), max_ms=g.lat_ms.max())
        row.update(mon); row.update(extra or {})
        out.append(row)
    ROWS.extend(out)
    return pd.DataFrame(out)

def run_step(scenario, level, n_vu, worker, extra=None, show=True, by_tag=True):
    mon = Monitor(); mon.start()
    try:
        df = arun(closed_loop(n_vu, CFG["DURATION"], CFG["WARMUP"], worker))
    finally:
        mon.stop()
    df.to_csv(RESULTS / f"raw_{scenario}_{level}.csv.gz", index=False)
    res = summarize(df, scenario, level, mon.summary(skip=CFG["WARMUP"]), extra, by_tag)
    if show:
        cols = ["name", "tag", "count", "rps", "err_pct", "p50_ms", "p95_ms", "p99_ms", "java_cores", "pg_cores", "client_cores", "lock_waiters_mean"]
        print(f"[{scenario} | level={level}]"); print(res[cols].round(2).to_string(index=False))
    return df, res

# ---- smoke test: đăng nhập thật + 1 request có xác thực ----
db("TRUNCATE users RESTART IDENTITY CASCADE")
HASH = "$2a$10$DuSMEmXqBUBN2MaSDuCix.pqSO30r5FflH..jSf2yTzCgygxHwqva"   # BCrypt("123456") lấy từ seed/seed.sql của repo
db("INSERT INTO users(email,password_hash,name,bio,created_at) VALUES ('smoke@bench.test',%s,'Smoke','x',now())", (HASH,))
import urllib.request as ur
req = ur.Request(BASE + "/api/v1/auth/login", data=json.dumps({"email": "smoke@bench.test", "password": "123456"}).encode(),
                 headers={"Content-Type": "application/json"})
t0 = time.perf_counter(); tk = json.loads(ur.urlopen(req).read())["access_token"]; login_ms = (time.perf_counter() - t0) * 1000
req = ur.Request(BASE + "/api/v1/recommendations?limit=5", headers={"Authorization": "Bearer " + tk})
assert ur.urlopen(req).status == 200
# token tự ký phải được app chấp nhận
req = ur.Request(BASE + "/api/v1/recommendations?limit=5", headers={"Authorization": "Bearer " + token(1)})
assert ur.urlopen(req).status == 200
print(f"OK: login thật {login_ms:.0f} ms (gồm BCrypt, lần đầu), JWT tự ký được chấp nhận.")

## 4. Nạp dữ liệu giả lập
Bố cục id (với `N_USERS = 50 000`):

| Vùng id | Dùng cho |
|---|---|
| 1 … 40% | **mục tiêu swipe / hồ sơ được gợi ý** (hot profile nằm ở các id thấp nhất) |
| 40% … 60% | người swipe cho S2/S4 (nửa đầu) và S6 (nửa sau) |
| 60% … 80% | S3 (đúng đắn của match) |
| 80% … 90% | S1 (người dùng "probe" có lịch sử swipe định sẵn) |
| 90% … 100% | S5 (kết nối SSE) |

Lịch sử swipe của probe ở S1 là **các id 1…H** (toàn `PASS`). Đây là mô hình *thực tế và cũng là xấu nhất*: API gợi ý theo `ORDER BY id`, nên người dùng luôn thấy hồ sơ nhỏ nhất chưa swipe, và sau H lần swipe thì tập đã swipe chính là 1…H.

In [ ]:
N = CFG["N_USERS"]
def rng_ids(a, b): return (int(a * N) + 1, int(b * N))
L = dict(targets=rng_ids(0.0, 0.4), pool=rng_ids(0.4, 0.6), s3=rng_ids(0.6, 0.8), s1=rng_ids(0.8, 0.9), s5=rng_ids(0.9, 1.0))
pool_mid = (L["pool"][0] + L["pool"][1]) // 2
L["pool_a"] = (L["pool"][0], pool_mid)          # S2, S4
L["pool_b"] = (pool_mid + 1, L["pool"][1])      # S6
T_MAX = L["targets"][1]
print(L)

db("TRUNCATE users RESTART IDENTITY CASCADE")
db('''INSERT INTO users(email,password_hash,name,bio,created_at)
      SELECT 'u'||g||'@bench.test', %s, 'User '||g, 'bio '||g, now() FROM generate_series(1,%s) g''', (HASH, N))

S1_LEVELS = [h for h in CFG["S1_LEVELS"] if h <= T_MAX]
S1_PROBES = {}
base = L["s1"][0]
for i, H in enumerate(S1_LEVELS):
    ids = list(range(base + i * CFG["S1_PROBES"], base + (i + 1) * CFG["S1_PROBES"]))
    assert ids[-1] <= L["s1"][1], "Vùng S1 quá nhỏ"
    S1_PROBES[H] = ids
    if H > 0:
        db('''INSERT INTO swipes(swiper_id,target_id,action,created_at)
              SELECT p, t, 'PASS', now() FROM unnest(%s::bigint[]) p CROSS JOIN generate_series(1,%s) t''', (ids, H))
db("ANALYZE")
print(db("select (select count(*) from users), (select count(*) from swipes)", fetch=True))

# ---- khởi động JVM/JIT/pool trước khi đo (dùng người dùng vùng S5, sẽ được dọn lại) ----
async def warm_worker(sess, rec, rng, i):
    lo5, hi5 = L["s5"]; uid = rng.randint(lo5, min(lo5 + 199, hi5))
    await call(sess, rec, "warm", "GET", "/api/v1/recommendations?limit=20", uid)
    await call(sess, rec, "warm", "POST", "/api/v1/swipes", uid, {"target_id": rng.randint(1, T_MAX), "action": "pass"}, ok=(201, 409))
_w = arun(closed_loop(16, 3 if QUICK else 20, 0, warm_worker))
print(f"Khởi động xong: {len(_w)} request, bỏ qua trong số liệu.")
db("DELETE FROM swipes WHERE swiper_id BETWEEN %s AND %s", L["s5"])
print(db("select version()", fetch=True)[0][0])

## 5. S1 – Chi phí gợi ý tăng theo lịch sử swipe
Mỗi mức H có 16 người dùng probe, mỗi người đã swipe `PASS` các id 1…H. Ta đo `GET /recommendations?limit=20` (đồng thời 8 client) và chạy thêm `EXPLAIN (ANALYZE, BUFFERS)` trên đúng câu SQL tương đương của `findRecommendations` để có bằng chứng ở tầng DB (thời gian thực thi, số block đã đọc).

**Cách đọc kết quả:** nếu p95 và `db_exec_ms` tăng gần tuyến tính theo H thì chi phí gợi ý phụ thuộc vào kích thước lịch sử – đúng vấn đề cần giải quyết ở Pha 2.

In [ ]:
EXPLAIN_SQL = '''EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)
select u.* from users u
where u.id <> %s and not exists (select 1 from swipes s where s.swiper_id = %s and s.target_id = u.id)
order by u.id limit 20'''

def explain_probe(uid, reps=5):
    ex, hits = [], []
    for _ in range(reps):
        x = db(EXPLAIN_SQL, (uid, uid), fetch=True)[0][0]
        x = json.loads(x) if isinstance(x, str) else x
        ex.append(x[0]["Execution Time"])
        p = x[0]["Plan"]; hits.append(p.get("Shared Hit Blocks", 0) + p.get("Shared Read Blocks", 0))
    return float(np.median(ex)), int(np.median(hits))

S1_RES = []
for H in S1_LEVELS:
    probes = S1_PROBES[H]
    async def w(sess, rec, rng, i, probes=probes, H=H):
        await call(sess, rec, "GET /recommendations", "GET", "/api/v1/recommendations?limit=20", rng.choice(probes), tag=f"H={H}")
    db_ms, blocks = explain_probe(probes[0])
    _, res = run_step("S1_reco_vs_history", H, CFG["S1_CONC"], w, extra=dict(history=H, db_exec_ms=db_ms, db_blocks=blocks))
    S1_RES.append(res)
S1 = pd.concat(S1_RES)
S1[["history", "rps", "p50_ms", "p95_ms", "p99_ms", "db_exec_ms", "db_blocks", "pg_cores", "java_cores"]].round(2)

## 6. S2 – Tải ghi: `POST /swipes` theo mức đồng thời
Mỗi request chọn ngẫu nhiên một người swipe trong `pool_a` và một mục tiêu **đều ngẫu nhiên** trong vùng `targets` (không lặp cặp để tránh 409). Tăng dần số client để tìm điểm bão hòa (RPS ngừng tăng, p95 tăng vọt). Nhớ rằng app dùng connection pool mặc định của Spring Boot (HikariCP, 10 kết nối) và Tomcat mặc định (200 thread); đó là cấu hình Pha 1 và **không** được đổi giữa các lần đo.

In [ ]:
USED = set()                       # các cặp (người swipe, mục tiêu) đã dùng, để không bị 409
HOT_K = CFG["S4_HOT_K"]

def uniform_sampler(rng): return rng.randint(1, T_MAX)

def make_swipe_worker(pool, sampler, like_p=0.5):
    async def w(sess, rec, rng, i):
        uid = tgt = None
        for _ in range(30):
            uid = rng.randint(*pool); tgt = sampler(rng)
            if (uid, tgt) not in USED:
                USED.add((uid, tgt)); break
        act = "like" if rng.random() < like_p else "pass"
        await call(sess, rec, "POST /swipes", "POST", "/api/v1/swipes", uid, {"target_id": tgt, "action": act},
                   ok=(201, 409), tag="hot" if tgt <= HOT_K else "cold")
    return w

S2_RES = []
for c in CFG["S2_CONC"]:
    _, res = run_step("S2_swipe_uniform", c, c, make_swipe_worker(L["pool_a"], uniform_sampler), extra=dict(concurrency=c), show=False, by_tag=False)
    S2_RES.append(res)
S2 = pd.concat(S2_RES)
S2[["concurrency", "count", "rps", "err_pct", "p50_ms", "p95_ms", "p99_ms", "java_cores", "pg_cores", "client_cores", "lock_waiters_mean"]].round(2)

## 7. S3 – Đúng đắn của match dưới tranh chấp
Ba thử nghiệm chạy đồng thời, lặp `S3_REPEATS` lần (race condition mang tính xác suất nên cần lặp):
1. **Cặp đối ứng**: A và B like nhau *cùng lúc*. Kỳ vọng: đúng 1 match, đúng 1 trong 2 response có `matched=true`.
2. **Swipe trùng**: cùng một A→B gửi 2 lần đồng thời. Kỳ vọng: đúng một `201` và một `409`.
3. **Hình sao**: một hub và nhiều người like nhau đồng thời (tranh chấp lock trên một dòng). Kỳ vọng: không deadlock, mỗi cặp đúng 1 match.

Sau đó kiểm tra bất biến trực tiếp trong DB. **Mọi con số vi phạm phải bằng 0.** Nếu khác 0, đó là bằng chứng cụ thể cho mục "đúng đắn của match" ở Pha 2.

In [ ]:
CHECKS = {
 "missing_match": '''SELECT count(*) FROM swipes s1 JOIN swipes s2 ON s1.swiper_id=s2.target_id AND s1.target_id=s2.swiper_id
    WHERE s1.action='LIKE' AND s2.action='LIKE' AND s1.swiper_id<s1.target_id AND s1.swiper_id BETWEEN %(lo)s AND %(hi)s
    AND NOT EXISTS (SELECT 1 FROM matches m WHERE m.user_a_id=s1.swiper_id AND m.user_b_id=s1.target_id)''',
 "orphan_match": '''SELECT count(*) FROM matches m WHERE m.user_a_id BETWEEN %(lo)s AND %(hi)s AND NOT EXISTS (
    SELECT 1 FROM swipes s1 JOIN swipes s2 ON s1.swiper_id=s2.target_id AND s1.target_id=s2.swiper_id
    WHERE s1.swiper_id=m.user_a_id AND s1.target_id=m.user_b_id AND s1.action='LIKE' AND s2.action='LIKE')''',
 "duplicate_pair": '''SELECT count(*) FROM (SELECT 1 FROM matches WHERE user_a_id BETWEEN %(lo)s AND %(hi)s
    GROUP BY least(user_a_id,user_b_id), greatest(user_a_id,user_b_id) HAVING count(*)>1) t''',
 "non_canonical_pair": "SELECT count(*) FROM matches WHERE user_a_id BETWEEN %(lo)s AND %(hi)s AND user_a_id>=user_b_id",
 "notification_mismatch": '''SELECT count(*) FROM matches m WHERE m.user_a_id BETWEEN %(lo)s AND %(hi)s
    AND (SELECT count(*) FROM notifications n WHERE n.match_id=m.id) <> 2''',
}

def clean_range(lo, hi):
    db("DELETE FROM notifications WHERE user_id BETWEEN %s AND %s", (lo, hi))
    db("DELETE FROM matches WHERE user_a_id BETWEEN %s AND %s", (lo, hi))
    db("DELETE FROM swipes WHERE swiper_id BETWEEN %s AND %s", (lo, hi))

def deadlocks():
    return db("select deadlocks from pg_stat_database where datname='heartlink'", fetch=True)[0][0]

async def s3_once():
    lo, hi = L["s3"]; ids = iter(range(lo, hi + 1))
    P, D, S = CFG["S3_PAIRS"], CFG["S3_DUP"], CFG["S3_STAR"]
    pairs = [(next(ids), next(ids)) for _ in range(P)]
    dups  = [(next(ids), next(ids)) for _ in range(D)]
    hub   = next(ids); spokes = [next(ids) for _ in range(S)]
    rec = []
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=120)) as sess:
        async def sw(a, b):
            st, data = await call(sess, rec, "POST /swipes", "POST", "/api/v1/swipes", a, {"target_id": b, "action": "like"}, ok=(201, 409), tag="s3")
            try: matched = bool(json.loads(data).get("matched"))
            except Exception: matched = False
            return st, matched
        r_pairs = await asyncio.gather(*[asyncio.gather(sw(a, b), sw(b, a)) for a, b in pairs],
                                       *[asyncio.gather(sw(a, b), sw(a, b)) for a, b in dups],
                                       asyncio.gather(*([sw(s, hub) for s in spokes] + [sw(hub, s) for s in spokes])))
    rp, rd, rs = r_pairs[:P], r_pairs[P:P + D], r_pairs[-1]
    v = {}
    v["pair_bad_status"]   = sum(1 for r in rp if any(st != 201 for st, _ in r))
    v["pair_matched_flag"] = sum(1 for r in rp if sum(m for _, m in r) != 1)       # kỳ vọng đúng 1 response matched=true
    v["dup_not_201_409"]   = sum(1 for r in rd if sorted(st for st, _ in r) != [201, 409])
    v["star_bad_status"]   = sum(1 for st, _ in rs if st != 201)
    v["star_matched_flag"] = abs(sum(m for _, m in rs) - S)                        # kỳ vọng đúng S response matched=true
    df = pd.DataFrame(rec, columns=COLS)
    return v, df

S3_ROWS = []
lo, hi = L["s3"]
for rep in range(CFG["S3_REPEATS"]):
    clean_range(lo, hi)
    d0 = deadlocks()
    v, df = arun(s3_once())
    v["deadlocks"] = deadlocks() - d0
    v["http_5xx_or_exc"] = int((~df.status.between(200, 499)).sum())
    for k, q in CHECKS.items():
        v["db_" + k] = db(q, {"lo": lo, "hi": hi}, fetch=True)[0][0]
    v["matches_created"] = db("select count(*) from matches where user_a_id between %s and %s", (lo, hi), fetch=True)[0][0]
    v["expected_matches"] = CFG["S3_PAIRS"] + CFG["S3_STAR"]
    v["p95_ms"] = pct(df.lat_ms, 95); v["rep"] = rep + 1
    S3_ROWS.append(v)
S3 = pd.DataFrame(S3_ROWS).set_index("rep")
S3.to_csv(RESULTS / "s3_correctness.csv")
viol_cols = [c for c in S3.columns if c not in ("matches_created", "expected_matches", "p95_ms")]
print("Tổng số vi phạm (kỳ vọng 0):", int(S3[viol_cols].sum().sum()))
S3.T

## 8. S4 – Hồ sơ nổi tiếng (phân phối Zipf)
Mục tiêu swipe lấy theo **Zipf** (id nhỏ = nổi tiếng nhất) ở cùng mức đồng thời với S2, để so sánh với baseline đều ngẫu nhiên. Vì `SwipeService` khóa dòng `users` của cả hai bên (`SELECT … FOR UPDATE`), các swipe vào cùng một hồ sơ nổi tiếng sẽ phải xếp hàng. Quan sát: latency nhóm `hot` so với `cold`, và `lock_waiters_*` (số tiến trình PostgreSQL đang chờ lock).

Lưu ý thêm từ code: `findRecommendations` sắp xếp theo `u.id`, nên **mọi người dùng mới đều được gợi ý cùng các id nhỏ nhất trước** – tự nó đã tạo ra hot profile ngay cả khi người dùng thật không "nổi tiếng". Kịch bản S6 phản ánh hiệu ứng này.

In [ ]:
ranks = np.arange(1, T_MAX + 1, dtype=float)
w_ = ranks ** (-CFG["S4_ZIPF"]); CDF = np.cumsum(w_ / w_.sum())
def zipf_sampler(rng): return int(min(np.searchsorted(CDF, rng.random()) + 1, T_MAX))

c = CFG["S4_CONC"]
print(f"Tỉ lệ yêu cầu rơi vào top-{HOT_K} hồ sơ theo Zipf(s={CFG['S4_ZIPF']}): {CDF[HOT_K-1]*100:.1f}%")
_, S4u = run_step("S4_baseline_uniform", c, c, make_swipe_worker(L["pool_a"], uniform_sampler), extra=dict(concurrency=c), show=False)
_, S4z = run_step("S4_zipf", c, c, make_swipe_worker(L["pool_a"], zipf_sampler), extra=dict(concurrency=c), show=False)
S4 = pd.concat([S4u.assign(workload="uniform"), S4z.assign(workload="zipf")])
S4[["workload", "tag", "count", "rps", "p50_ms", "p95_ms", "p99_ms", "err_pct", "lock_waiters_mean", "lock_waiters_max", "pg_cores"]].round(2)

## 9. S5 – Độ trễ thông báo match qua SSE
Mở `C` kết nối `GET /notifications/stream` (mỗi kết nối một người dùng), sau đó cho các cặp like nhau với tốc độ `S5_RATE` cặp/giây. Đo từ lúc gửi like thứ hai (tạo match) đến lúc **mỗi** bên nhận event `match.created`, cùng tỉ lệ sự kiện bị mất và tài nguyên của JVM theo số kết nối.

Lưu ý: repo Pha 1 **đã có SSE** (best-effort, 1 instance, không replay) – tức là mục "thiếu push" trong tài liệu cần được diễn đạt lại theo hiện trạng (push có nhưng không bền, không chạy được nhiều instance). Kịch bản này đo đúng hiện trạng đó. Để so sánh với polling, độ trễ trung bình khi poll mỗi `T` giây ≈ `T/2` (tối đa `T`).

In [ ]:
async def s5_level(C):
    lo, hi = L["s5"]; users = list(range(lo, lo + C)); assert users[-1] <= hi, "Vùng S5 quá nhỏ cho C kết nối"
    npairs = min(CFG["S5_PAIRS"], C // 2); pairs = [(users[2 * i], users[2 * i + 1]) for i in range(npairs)]
    arrivals, ready, failed = {}, [0], [0]
    rec = []
    sse_sess = aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=None, sock_read=None))
    async def reader(uid):
        try:
            async with sse_sess.get(BASE + "/api/v1/notifications/stream",
                                    headers={"Authorization": "Bearer " + token(uid), "Accept": "text/event-stream"}) as r:
                if r.status != 200: failed[0] += 1; return
                ev = None
                async for raw in r.content:
                    line = raw.decode("utf-8", "ignore").strip()
                    if line.startswith(":") and "connected" in line: ready[0] += 1
                    elif line.startswith("event:"): ev = line[6:].strip()
                    elif line.startswith("data:") and ev == "match.created":
                        arrivals[(uid, json.loads(line[5:])["match_id"])] = time.perf_counter()
        except asyncio.CancelledError: raise
        except Exception: failed[0] += 1
    readers = [asyncio.create_task(reader(u)) for u in users]
    t0 = time.time()
    while ready[0] + failed[0] < C and time.time() - t0 < 90: await asyncio.sleep(0.2)
    await asyncio.sleep(1.0)
    java_thr = psutil.Process(APP.pid).num_threads(); java_rss = psutil.Process(APP.pid).memory_info().rss / 2**20

    sent = {}   # match_id -> (a, b, t_send)
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=60)) as sess:
        async def trigger(a, b):
            await call(sess, rec, "POST /swipes", "POST", "/api/v1/swipes", a, {"target_id": b, "action": "like"}, ok=(201, 409), tag="first")
            t_send = time.perf_counter()
            st, data = await call(sess, rec, "POST /swipes", "POST", "/api/v1/swipes", b, {"target_id": a, "action": "like"}, ok=(201, 409), tag="second")
            try: sent[json.loads(data)["match"]["id"]] = (a, b, t_send)
            except Exception: pass
        tasks = []
        for a, b in pairs:
            tasks.append(asyncio.create_task(trigger(a, b))); await asyncio.sleep(1.0 / CFG["S5_RATE"])
        await asyncio.gather(*tasks)
    await asyncio.sleep(8.0)    # chờ các event đến muộn
    delays, lost = [], 0
    for mid, (a, b, ts) in sent.items():
        for u in (a, b):
            t = arrivals.get((u, mid))
            if t is None: lost += 1
            else: delays.append((t - ts) * 1000.0)
    for t in readers: t.cancel()
    await asyncio.gather(*readers, return_exceptions=True); await sse_sess.close()
    n_exp = 2 * len(sent)
    return dict(connections=C, connected_ok=ready[0], connect_failed=failed[0], matches=len(sent), events_expected=n_exp,
                events_lost=lost, loss_pct=100.0 * lost / max(n_exp, 1),
                delay_p50_ms=pct(delays, 50), delay_p95_ms=pct(delays, 95), delay_p99_ms=pct(delays, 99), delay_max_ms=max(delays) if delays else float("nan"),
                java_threads=java_thr, java_rss_mb=java_rss)

S5_ROWS = []
lo, hi = L["s5"]
for C in CFG["S5_CONNS"]:
    clean_range(lo, hi)
    mon = Monitor(); mon.start()
    try: row = arun(s5_level(C))
    finally: mon.stop()
    row.update({k: v for k, v in mon.summary().items() if k in ("java_cores", "pg_cores", "client_cores")})
    S5_ROWS.append(row); print(row)
S5 = pd.DataFrame(S5_ROWS); S5.to_csv(RESULTS / "s5_sse.csv", index=False)
S5.round(2)

## 10. S6 – Phiên người dùng hỗn hợp (baseline tổng thể)
Mỗi vòng của một client: `GET /recommendations?limit=20` → swipe 5 hồ sơ đầu tiên được gợi ý (40% like, 60% pass) → 20% xác suất `GET /matches`. Vì swipe đúng những hồ sơ được gợi ý, lịch sử của người dùng tăng dần theo thứ tự id và các id nhỏ nhận nhiều swipe nhất, giống hành vi thật của API hiện tại. Đây là **con số chính để so sánh Pha 1 với Pha 2**.

In [ ]:
async def session_worker(sess, rec, rng, i):
    uid = rng.randint(*L["pool_b"])
    st, data = await call(sess, rec, "GET /recommendations", "GET", "/api/v1/recommendations?limit=20", uid)
    items = []
    if st == 200:
        try: items = json.loads(data)["items"]
        except Exception: pass
    for it in items[:5]:
        act = "like" if rng.random() < 0.4 else "pass"
        await call(sess, rec, "POST /swipes", "POST", "/api/v1/swipes", uid, {"target_id": it["id"], "action": act}, ok=(201, 409))
    if rng.random() < 0.2:
        await call(sess, rec, "GET /matches", "GET", "/api/v1/matches?limit=20&offset=0", uid)

S6_RES = []
for c in CFG["S6_CONC"]:
    _, res = run_step("S6_mixed_sessions", c, c, session_worker, extra=dict(concurrency=c), show=False, by_tag=False)
    S6_RES.append(res)
S6 = pd.concat(S6_RES)
S6[["concurrency", "name", "count", "rps", "err_pct", "p50_ms", "p95_ms", "p99_ms", "java_cores", "pg_cores", "client_cores", "lock_waiters_mean"]].round(2)

## 11. S7 – Chat theo match (mới trong `HEART_LINK`)
Code mới thêm `POST/GET /api/v1/matches/{id}/messages`. Gửi tin chạy `SELECT … FOR UPDATE` trên dòng `matches` (để nối tiếp với unmatch) và giữ khóa đến hết transaction. Chat chỉ có **polling**, chưa có realtime. Năm phép đo:
- **S7a**: gửi tin rải đều trên nhiều match (baseline).
- **S7b**: mọi client gửi vào **một** match → các lần gửi bị khóa nối tiếp. So RPS/p95 với S7a để thấy chi phí khóa theo match.
- **S7c**: `GET messages` (trang mới nhất và trang sâu bằng `before`) theo kích thước lịch sử.
- **S7d**: đúng đắn khi unmatch đồng thời với gửi tin: mọi tin **bắt đầu sau khi** DELETE đã trả về phải bị từ chối (404); không được có 5xx.
- **S7e**: tải polling: P người dùng cùng gọi `GET messages` mỗi `S7_POLL_S` giây (tải lý tưởng = P / chu kỳ RPS). Đây là mốc để Pha 2 so với cách phát tin thời gian thực.

In [ ]:
lo5, hi5 = L["s5"]
M = max(0, min(CFG["S7_MATCHES"], (hi5 - (lo5 + max(CFG["S5_CONNS"]))) // 2))
n_hist = len(CFG["S7_HIST"]) * 8
assert M >= 1 + n_hist + CFG["S7_CANCEL"] + 5, f"Chỉ tạo được {M} match, không đủ cho S7 – tăng N_USERS hoặc giảm S7_*"
clean_range(lo5, hi5)
c0 = hi5 - 2 * M + 1
db("INSERT INTO matches(user_a_id,user_b_id,status,created_at) SELECT g, g+1, 'ACTIVE', now() FROM generate_series(%s,%s,2) g", (c0, hi5 - 1))
CH = db("select id,user_a_id,user_b_id from matches where user_a_id>=%s order by id", (c0,), fetch=True)
HOT = CH[0]
HIST = {H: CH[1 + 8 * i: 9 + 8 * i] for i, H in enumerate(CFG["S7_HIST"])}
CANCEL_SET = CH[-CFG["S7_CANCEL"]:]
SPREAD = CH[1 + n_hist: -CFG["S7_CANCEL"]]
print(f"S7: {M} match | spread={len(SPREAD)} hist={n_hist} cancel={len(CANCEL_SET)}")

def msg_body(): return {"content": "xin chào " + "x" * 40}

def send_worker_for(matches):
    async def w(sess, rec, rng, i):
        mid, a, b = matches[rng.randrange(len(matches))] if len(matches) > 1 else matches[0]
        await call(sess, rec, "POST /messages", "POST", f"/api/v1/matches/{mid}/messages", a if rng.random() < .5 else b, msg_body(), ok=(201,))
    return w

c = CFG["S7_CONC"]
_, S7a = run_step("S7a_send_spread", c, c, send_worker_for(SPREAD), extra=dict(concurrency=c), show=False, by_tag=False)
_, S7b = run_step("S7b_send_hot_match", c, c, send_worker_for([HOT]), extra=dict(concurrency=c), show=False, by_tag=False)

# ---- S7c: đọc lịch sử theo kích thước ----
S7c_RES = []
for H, ms in HIST.items():
    if H > 0:
        db("INSERT INTO messages(match_id,sender_id,content,created_at) SELECT m.id, m.user_a_id, 'tin '||g, now() FROM matches m, generate_series(1,%s) g WHERE m.id = ANY(%s)", (H, [m[0] for m in ms]))
    cur_ids = {m[0]: db("select coalesce(min(id),0), coalesce(max(id),0) from messages where match_id=%s", (m[0],), fetch=True)[0] for m in ms}
    async def w(sess, rec, rng, i, ms=ms, H=H, cur_ids=cur_ids):
        mid, a, b = ms[rng.randrange(len(ms))]
        await call(sess, rec, "GET /messages latest", "GET", f"/api/v1/matches/{mid}/messages?limit=20", a, tag=f"H={H}")
        lo_id, hi_id = cur_ids[mid]
        if H > 0:
            await call(sess, rec, "GET /messages deep", "GET", f"/api/v1/matches/{mid}/messages?limit=20&before={(lo_id + hi_id) // 2}", a, tag=f"H={H}")
    _, res = run_step("S7c_list_history", H, c, w, extra=dict(history=H), show=False)
    S7c_RES.append(res)
S7c = pd.concat(S7c_RES)

# ---- S7d: unmatch đồng thời với gửi tin ----
async def s7d_once():
    rec = []; viol = dict(sent_after_unmatch_accepted=0, bad_status=0, unmatch_failed=0); chk = dict(n=0)
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=0), timeout=aiohttp.ClientTimeout(total=120)) as sess:
        async def one(mid, a, b):
            t_del_end = [None]
            async def deleter():
                await asyncio.sleep(random.random() * 0.03)
                st, _ = await call(sess, rec, "DELETE /matches", "DELETE", f"/api/v1/matches/{mid}", a, ok=(204,))
                t_del_end[0] = time.perf_counter()
                if st != 204: viol["unmatch_failed"] += 1
            async def sender(k):
                for j in range(6):
                    t_start = time.perf_counter()
                    st, _ = await call(sess, rec, "POST /messages", "POST", f"/api/v1/matches/{mid}/messages", b if (k + j) % 2 else a, msg_body(), ok=(201, 404))
                    if st not in (201, 404): viol["bad_status"] += 1
                    if st == 201 and t_del_end[0] is not None and t_start > t_del_end[0]: viol["sent_after_unmatch_accepted"] += 1
                    if t_del_end[0] is not None and t_start > t_del_end[0]: chk["n"] += 1
            await asyncio.gather(deleter(), *[sender(k) for k in range(4)])
        await asyncio.gather(*[one(*m) for m in CANCEL_SET])
    viol["requests_after_unmatch_checked"] = chk["n"]
    return viol, pd.DataFrame(rec, columns=COLS)

v7d, df7d = arun(s7d_once())
v7d["http_5xx_or_exc"] = int((~df7d.status.between(200, 499)).sum())
S7d = pd.Series(v7d, name="S7d"); S7d.to_csv(RESULTS / "s7d_unmatch_consistency.csv")
print("S7d (kỳ vọng các mục vi phạm = 0):", v7d)

# ---- S7e: tải polling ----
S7e_RES = []
for P in CFG["S7_POLLERS"]:
    started = set()
    async def w(sess, rec, rng, i):
        if i not in started:
            started.add(i); await asyncio.sleep(rng.random() * CFG["S7_POLL_S"])
        mid, a, b = SPREAD[i % len(SPREAD)]
        await call(sess, rec, "GET /messages poll", "GET", f"/api/v1/matches/{mid}/messages?limit=20", a)
        await asyncio.sleep(CFG["S7_POLL_S"])
    _, res = run_step("S7e_polling", P, P, w, extra=dict(pollers=P, ideal_rps=P / CFG["S7_POLL_S"]), show=False, by_tag=False)
    S7e_RES.append(res)
S7e = pd.concat(S7e_RES)
print(S7e[["pollers", "ideal_rps", "rps", "err_pct", "p50_ms", "p95_ms", "p99_ms", "java_cores", "pg_cores", "client_cores"]].round(2).to_string(index=False))
S7ab = pd.concat([S7a.assign(case="spread"), S7b.assign(case="hot_match")])
S7ab[["case", "count", "rps", "err_pct", "p50_ms", "p95_ms", "p99_ms", "lock_waiters_mean", "lock_waiters_max", "pg_cores"]].round(2)

## 12. Tổng hợp: bảng kết quả, biểu đồ, môi trường đo
Mọi thứ được lưu trong `/kaggle/working/results/` (`summary.csv`, `env.json`, các biểu đồ PNG, file raw nén) và đóng gói thành `results_phase1.zip` để tải về/commit vào repo. **Pha 2 chạy lại đúng notebook này với cùng cấu hình** rồi so sánh với `summary.csv` của Pha 1.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

SUMMARY = pd.DataFrame(ROWS)
SUMMARY.to_csv(RESULTS / "summary.csv", index=False)

env_info = dict(
    when=time.strftime("%Y-%m-%d %H:%M:%S"), quick_mode=QUICK, commit=COMMIT, cfg=CFG, platform=platform.platform(),
    cpu_count=os.cpu_count(), lscpu=sh("lscpu | egrep 'Model name|^CPU\\(s\\)|Thread|MHz' || true", check=False),
    mem=sh("free -m | head -2", check=False), java=sh("java -version 2>&1 | head -1", check=False),
    postgres=db("select version()", fetch=True)[0][0],
    app_defaults="Spring Boot defaults: HikariCP pool=10, Tomcat threads=200 (không đổi giữa Pha 1 và Pha 2)",
)
(RESULTS / "env.json").write_text(json.dumps(env_info, indent=2, ensure_ascii=False))
print(env_info["lscpu"]); print(env_info["mem"])

def savefig(name):
    plt.tight_layout(); plt.savefig(RESULTS / name, dpi=130); plt.show()

# S1
if not S1.empty:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    x = S1.history.astype(int)
    ax[0].plot(x, S1.p50_ms, "o-", label="p50"); ax[0].plot(x, S1.p95_ms, "o-", label="p95"); ax[0].plot(x, S1.p99_ms, "o-", label="p99")
    ax[0].set(xlabel="Số swipe đã có của người dùng (H)", ylabel="Latency GET /recommendations (ms)", title="S1: API gợi ý"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(x, S1.db_exec_ms, "s-", color="C3"); ax[1].set(xlabel="H", ylabel="Thời gian thực thi SQL (ms)", title="S1: EXPLAIN ANALYZE"); ax[1].grid(alpha=.3)
    savefig("s1_recommendation.png")

# S2 / S6
for sc, df_, title in (("S2", S2, "S2: POST /swipes (đều ngẫu nhiên)"), ("S6", S6[S6.name == "GET /recommendations"] if not S6.empty else S6, "S6: phiên hỗn hợp – GET /recommendations")):
    if df_.empty: continue
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    if sc == "S6":
        tot = S6.groupby("concurrency").agg(rps=("rps", "sum")).reset_index()
        ax[0].plot(tot.concurrency, tot.rps, "o-"); ax[0].set(ylabel="Tổng RPS (mọi endpoint)")
    else:
        ax[0].plot(df_.concurrency, df_.rps, "o-"); ax[0].set(ylabel="RPS")
    ax[0].set(xlabel="Số client đồng thời", title=title); ax[0].grid(alpha=.3)
    for q, lab in (("p50_ms", "p50"), ("p95_ms", "p95"), ("p99_ms", "p99")): ax[1].plot(df_.concurrency, df_[q], "o-", label=lab)
    ax[1].set(xlabel="Số client đồng thời", ylabel="Latency (ms)", title="Phân vị latency"); ax[1].legend(); ax[1].grid(alpha=.3)
    savefig(f"{sc.lower()}_load.png")

# S4
if not S4.empty:
    t = S4.pivot_table(index="workload", columns="tag", values=["p50_ms", "p95_ms", "p99_ms"])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    t["p95_ms"].plot.bar(ax=ax[0], rot=0); ax[0].set(ylabel="p95 (ms)", title="S4: p95 theo loại mục tiêu"); ax[0].grid(alpha=.3, axis="y")
    S4.groupby("workload")[["lock_waiters_mean", "lock_waiters_max"]].first().plot.bar(ax=ax[1], rot=0); ax[1].set(ylabel="Số tiến trình PG chờ lock", title="S4: tranh chấp lock"); ax[1].grid(alpha=.3, axis="y")
    savefig("s4_hot_profile.png")

# S5
if not S5.empty:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(S5.connections, S5.delay_p50_ms, "o-", label="p50"); ax[0].plot(S5.connections, S5.delay_p95_ms, "o-", label="p95"); ax[0].plot(S5.connections, S5.delay_p99_ms, "o-", label="p99")
    ax[0].set(xlabel="Số kết nối SSE", ylabel="Độ trễ thông báo (ms)", title="S5: gửi like → nhận event"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(S5.connections, S5.java_rss_mb, "o-", color="C3"); ax[1].set(xlabel="Số kết nối SSE", ylabel="RSS của JVM (MB)", title="S5: bộ nhớ theo số kết nối"); ax[1].grid(alpha=.3)
    savefig("s5_sse.png")

# S7
if 'S7ab' in globals() and not S7ab.empty:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    S7ab.set_index("case")[["rps"]].plot.bar(ax=ax[0], rot=0, legend=False); ax[0].set(ylabel="RPS", title="S7: gửi tin – rải đều vs một match"); ax[0].grid(alpha=.3, axis="y")
    S7ab.set_index("case")[["p50_ms", "p95_ms", "p99_ms"]].plot.bar(ax=ax[1], rot=0); ax[1].set(ylabel="Latency (ms)", title="S7: phân vị latency"); ax[1].grid(alpha=.3, axis="y")
    savefig("s7_chat_send.png")

# bảng tổng quan
headline = SUMMARY[SUMMARY.scenario.isin(["S2_swipe_uniform", "S6_mixed_sessions", "S7a_send_spread", "S7b_send_hot_match", "S7e_polling"])][["scenario", "level", "name", "rps", "err_pct", "p95_ms", "p99_ms", "java_cores", "pg_cores", "client_cores"]].round(2)
print("\n=== Bảng tổng quan ===\n", headline.to_string(index=False))
if QUICK: print("\n!!! QUICK mode: số liệu chỉ để kiểm tra pipeline, không dùng để báo cáo.")

shutil.make_archive(str(WORK / "results_phase1"), "zip", RESULTS)
print("Đã đóng gói:", WORK / "results_phase1.zip")

In [ ]:
# Dọn dẹp (tùy chọn): dừng app và PostgreSQL
APP.terminate()
try: APP.wait(timeout=20)
except Exception: APP.kill()
sh(f"su postgres -c '{PGBIN}/pg_ctl -D {PGDATA} -m fast stop'", check=False)